# Project 01 (basic) — DQN from scratch: balancing CartPole

> **Module 14 — Deep Reinforcement Learning** · Format: **Jupyter notebook (PyTorch)**
>
> **Why this format?** Deep RL lives off *watching*: learning curves that jitter, a pole that
> eventually stays up. A notebook connects code, training and visualization — ideal for your
> first own **deep Q-network** agent.

## Goal
You build a **DQN** (Mnih et al. 2015 — the "Atari paper", here in miniature) and use it to teach
an agent to solve the **CartPole** task: balancing a pole on a movable cart **upright** through
left/right pushes. We build the environment (the physics) **ourselves** — entirely without `gym`.

You implement the **two conceptual core pieces**; the rest (physics, replay buffer, the network,
the training loop, plots) is given:
1. the **ε-greedy action selection** (`select_action`),
2. the **DQN target & loss** (`learn`) — the heart of the algorithm.

## Prior knowledge
Script module 14, section **2** (DQN, experience replay, target network). Module 13 (Q-learning,
ε-greedy). Module 05 (PyTorch: `nn.Module`, Adam, backprop).

## What should work in the end
A learning curve that rises from ~15 to several hundred steps, and a **greedy evaluation** in
which the agent holds the pole for the full episode length (500 steps). Training: ~20–40 s on the
CPU.

> **How to work:** fill in the `# TODO` places yourself. The reference solution is in
> `solution/dqn_cartpole_solution.ipynb`.


In [ ]:
import math, random, time
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# reproducibility
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cpu")   # small networks -> CPU is fast enough here (and reproducible)
print("torch", torch.__version__, "| device:", device)

## 1 · The environment: CartPole by hand

The state is $s=[x,\dot x,\theta,\dot\theta]$ (cart position & velocity, pole angle & angular
velocity). Two actions: **0 = push left**, **1 = push right**. Reward **+1 per step** in which
the pole is still up. The episode ends when the cart drives too far ($|x|>2.4$), the pole tips too
far ($|\theta|>12°$) or 500 steps are reached. The dynamics are the classical (Euler-integrated)
pole-on-cart physics — fully given.


In [ ]:
class CartPole:
    def __init__(self):
        self.g=9.8; self.mc=1.0; self.mp=0.1; self.l=0.5; self.fmag=10.0; self.tau=0.02
        self.mt=self.mc+self.mp; self.pml=self.mp*self.l
        self.x_thr=2.4; self.th_thr=12*math.pi/180; self.max_steps=500
        self.n_states=4; self.n_actions=2

    def reset(self):
        self.s=np.random.uniform(-0.05,0.05,4); self.steps=0
        return self.s.copy()

    def step(self, a):
        x,xd,th,thd=self.s
        f=self.fmag if a==1 else -self.fmag
        ct=math.cos(th); st=math.sin(th)
        temp=(f+self.pml*thd*thd*st)/self.mt
        thacc=(self.g*st-ct*temp)/(self.l*(4/3-self.mp*ct*ct/self.mt))
        xacc=temp-self.pml*thacc*ct/self.mt
        x+=self.tau*xd; xd+=self.tau*xacc; th+=self.tau*thd; thd+=self.tau*thacc
        self.s=np.array([x,xd,th,thd]); self.steps+=1
        done=bool(abs(x)>self.x_thr or abs(th)>self.th_thr or self.steps>=self.max_steps)
        return self.s.copy(), 1.0, done

# a short demo: a random policy falls over quickly
env=CartPole(); s=env.reset(); steps=0; done=False
while not done:
    s,r,done=env.step(random.randint(0,1)); steps+=1
print("random policy holds", steps, "steps (out of max. 500)")

## 2 · Q-network and replay buffer

**Q-network:** a small MLP $s\;(4) \to 128 \to 128 \to Q(s,\cdot)\;(2)$ — one Q value per action.
**Replay buffer:** a ring buffer that stores transitions $(s,a,r,s',\text{done})$; we train on
**random minibatches** from it (decorrelates the data). Both given.


In [ ]:
def make_qnet():
    return nn.Sequential(
        nn.Linear(4,128), nn.ReLU(),
        nn.Linear(128,128), nn.ReLU(),
        nn.Linear(128,2),
    ).to(device)

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buf=deque(maxlen=capacity)
    def push(self, s,a,r,s2,done):
        self.buf.append((s,a,r,s2,done))
    def sample(self, batch):
        bt=random.sample(self.buf, batch)
        s =torch.tensor(np.array([b[0] for b in bt]),dtype=torch.float32,device=device)
        a =torch.tensor([b[1] for b in bt],device=device).unsqueeze(1)
        r =torch.tensor([b[2] for b in bt],dtype=torch.float32,device=device).unsqueeze(1)
        s2=torch.tensor(np.array([b[3] for b in bt]),dtype=torch.float32,device=device)
        d =torch.tensor([float(b[4]) for b in bt],device=device).unsqueeze(1)
        return s,a,r,s2,d
    def __len__(self): return len(self.buf)

## 3 · The DQN agent — **this is your work**

Two methods are to be filled in:

**`select_action(state)` — ε-greedy** (as in module 13, only with the network instead of the
table): with probability $\varepsilon$ a random action, otherwise $\arg\max_a Q(s,a;\theta)$.
Tip: for the network use `torch.tensor(state, dtype=torch.float32)` and `with torch.no_grad():`.

**`learn()` — the DQN update.** The batch is already drawn. Fill in the **target** and the
**loss**:
- **Target** (with the **target network** $\theta^-$, no gradient!):
  $$y = r + \gamma\,(1-\text{done})\,\max_{a'} Q(s',a';\theta^-).$$
- **Prediction:** $Q(s,a;\theta)$ — the Q value of the *actually chosen* action (`gather`).
- **Loss:** `smooth_l1_loss`(prediction, target) (Huber — more robust than MSE).

The **target network** is afterwards pulled along gently via a **Polyak average** (given) — that
stabilizes learning (script 2.1: "do not shoot at a moving target").


In [ ]:
class DQNAgent:
    def __init__(self, gamma=0.99, lr=1e-3, batch=128, eps_start=1.0,
                 eps_min=0.02, eps_decay=0.99, tau=0.01):
        self.q  = make_qnet()
        self.qt = make_qnet(); self.qt.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
        self.gamma=gamma; self.batch=batch; self.tau=tau
        self.eps=eps_start; self.eps_min=eps_min; self.eps_decay=eps_decay

    def select_action(self, state):
        # TODO: ε-greedy. With prob. self.eps a random action (0 or 1),
        #       otherwise argmax_a Q(state, a) from self.q.
        #   with torch.no_grad():
        #       q = self.q(torch.tensor(state, dtype=torch.float32, device=device))
        #       return int(q.argmax())
        raise NotImplementedError

    def learn(self):
        if len(self.buffer) < 1000:      # only learn once there is enough experience
            return None
        s,a,r,s2,d = self.buffer.sample(self.batch)

        # prediction Q(s,a;theta) for the actually chosen action a:
        # TODO: q_sa = self.q(s).gather(1, a)
        q_sa = None  # TODO

        # target y = r + gamma*(1-done)*max_a' Q(s',a';theta^-)   (target network, NO gradient!)
        with torch.no_grad():
            # TODO: y = r + self.gamma * (1 - d) * self.qt(s2).max(1, keepdim=True)[0]
            y = None  # TODO

        # Huber loss between the prediction and the target:
        # TODO: loss = nn.functional.smooth_l1_loss(q_sa, y)
        loss = None  # TODO

        self.opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.opt.step()

        # pull the target network along gently (Polyak):  theta^- <- (1-tau) theta^- + tau theta
        with torch.no_grad():
            for p, pt in zip(self.q.parameters(), self.qt.parameters()):
                pt.mul_(1 - self.tau).add_(self.tau * p)
        return float(loss)

    def decay_epsilon(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)


## 4 · Training

The loop is given: per step choose an action (ε-greedy), advance the environment step by step,
push the transition into the buffer, **one** learning step. After each episode lower ε.

Two robustness ingredients (standard practice, given here):
- **Best-model checkpoint** — every few episodes a short **greedy** evaluation; the best weight
  version so far is stored and reloaded at the end. This way the result does **not** depend on
  where the (jittery) training happens to end — DQN tends towards *catastrophic forgetting* (see
  the conclusion).
- **Solved criterion** — if the greedy evaluation reaches ~500, the task is solved → stop.

> We set the seeds again directly before training, so that this run is **reproducible**,
> independent of the cells before it.


In [ ]:
import copy

def greedy_score(agent, env, episodes=5):
    saved=agent.eps; agent.eps=0.0
    res=[]
    for _ in range(episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s); s,r,done=env.step(a); total+=r
        res.append(total)
    agent.eps=saved
    return float(np.mean(res))

def train(agent, env, n_episodes=500, eval_every=10, solved=490.0, verbose=True):
    scores=[]; best=-1.0; best_weights=None; t0=time.time()
    for ep in range(n_episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s)
            s2,r,done=env.step(a)
            agent.buffer.push(s,a,r,s2,done)
            agent.learn()
            s=s2; total+=r
        agent.decay_epsilon()
        scores.append(total)
        if verbose and ep%20==0:
            print(f"ep {ep:3d}  score {total:5.0f}  avg20 {np.mean(scores[-20:]):6.1f}"
                  f"  eps {agent.eps:.2f}  t {time.time()-t0:.0f}s")
        # regular greedy evaluation -> remember the best model, possibly stop early
        if ep>=30 and ep%eval_every==0:
            g=greedy_score(agent, env, episodes=10)
            if g>best:
                best=g; best_weights=copy.deepcopy(agent.q.state_dict())
            if best>=solved:
                print(f"Solved: greedy score {best:.0f} at episode {ep}.")
                break
    if best_weights is not None:
        agent.q.load_state_dict(best_weights)      # reload the best model
    print(f"Training done in {time.time()-t0:.0f}s. Best greedy score: {best:.0f}")
    return scores

# make it reproducible (independent of the cells before)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
agent = DQNAgent()
scores = train(agent, CartPole(), n_episodes=500)

In [ ]:
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode="valid")

plt.figure(figsize=(8,4.2))
plt.plot(scores, alpha=0.3, label="episode score")
plt.plot(range(19, len(scores)), moving_average(scores,20), lw=2, label="moving average (20)")
plt.axhline(500, ls="--", color="gray", lw=1, label="maximum (500)")
plt.xlabel("episode"); plt.ylabel("steps balanced (= return)")
plt.title("DQN learns CartPole"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5 · Greedy evaluation

The training scores still contain ε-exploration. The true test: the **greedy** policy (ε=0) over
several episodes. A well-trained agent holds the pole for the full **500** steps.


In [ ]:
def evaluate(agent, env, episodes=10):
    saved=agent.eps; agent.eps=0.0            # purely greedy
    results=[]
    for _ in range(episodes):
        s=env.reset(); done=False; total=0.0
        while not done:
            a=agent.select_action(s); s,r,done=env.step(a); total+=r
        results.append(total)
    agent.eps=saved
    return results

ev=evaluate(agent, CartPole(), episodes=10)
print("Greedy evaluation (10 episodes):", ev)
print("Mean:", np.mean(ev))

## 6 · Observations & conclusion

- The learning curve is **not monotonic** — typical for DQN. It often rises, **drops**
  (*catastrophic forgetting*: the network briefly "unlearns") and recovers again. This is a direct
  symptom of the **deadly triad** (script 1.2): function approximation + bootstrapping +
  off-policy. **Replay** and the **target network** tame it far enough that the greedy agent
  reliably holds the full 500 steps at the end.
- Despite the jittery training curve, the **greedy policy** is usually perfect at the end — the
  training score (with exploration) and the true performance (greedy) are simply different things.

### Mini-tasks
1. **Without the target network:** set `tau=1.0` (target = the online network every step). Does
   the training become more unstable? (That is the reason for the target network.)
2. **Double DQN** (script 2.2): change the target to
   $y=r+\gamma\,Q(s',\arg\max_{a'}Q(s',a';\theta);\theta^-)$ — the action from the online network,
   the evaluation from the target network. Does that reduce the overestimation?
3. A smaller network (e.g. $64$) or a different $\varepsilon$-decay — how robust is the learning?
4. Plot the mean **Q values** during training — do they ever diverge?

> Write down your answers first, then compare them with the reference answers in the `solution/`
> notebook.
